# 03 — Modelos lineares com engenharia de atributos

**Hipótese:** A afinidade, sua curvatura e a confiança nas médias explicam parte do erro que os atributos brutos deixam passar. Ridge, Lasso e Elastic Net controlam a complexidade.

Compare primeiro Ridge bruto e Ridge com afinidade. Depois avalie o conjunto de engenharia e o efeito do encolhimento. A comparação entre Lasso e Ridge considera regularização e engenharia em conjunto; não isola cada atributo.

Este notebook usa somente o treino. As receitas e a execução compartilhada estão em `src/experiments.py`.

## 1. Dados e configuração

Execute as células em ordem. `SMOKE_TEST = True` verifica a execução com uma amostra pequena; seus números não são resultados de desempenho. A execução completa usa 15 dobras externas e busca interna com 3 dobras, podendo levar vários minutos.

In [2]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = Path.cwd().resolve()
if not (ROOT / "src").is_dir():
    ROOT = ROOT.parent
if not (ROOT / "src" / "data.py").is_file():
    raise RuntimeError("Abra o notebook a partir da raiz do projeto ou da pasta notebooks.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.data import load_train
from src.experiments import (recipes, evaluate_family, summary, save_results,
                             plot_diagnostics, load_results, search)
X, y = load_train()
pd.set_option("display.precision", 4)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})
print(f"Treino: {len(y)} pares; {X.shape[1]} atributos brutos.")

Treino: 4000 pares; 10 atributos brutos.


In [1]:
FAMILY = "lineares"
SMOKE_TEST = False

for name, (_, grid) in recipes(FAMILY).items():
    print(name, "→", grid)

NameError: name 'recipes' is not defined

## 2. Ajuste e avaliação

Em cada dobra externa, a busca escolhe os hiperparâmetros usando somente o treino dessa dobra. A função de seleção e a avaliação usam RMSE com corte em [0, 5]. Transformações, médias do encolhimento e escalas são aprendidas dentro do pipeline.

Os baselines constante e linear com afinidade usam as mesmas partições. As famílias não lineares recebem atributos construídos, incluindo afinidade: a comparação mede o pipeline completo.

In [ ]:
payload = evaluate_family(FAMILY, X, y, smoke=SMOKE_TEST)
table = summary(payload)
display(table)

## 3. Hiperparâmetros e análise dos erros

Confira a estabilidade das escolhas entre dobras e a diferença entre treino e validação. Os resíduos usam a média das previsões externas das três repetições; servem para diagnóstico, enquanto a tabela reporta a média dos 15 RMSEs. O desvio entre dobras é descritivo, não um intervalo de confiança: as dobras compartilham dados.

In [ ]:
choices = []
for record in payload["records"]:
    for fold, params in enumerate(record["best_params_by_fold"], start=1):
        choices.append({"modelo": record["name"], "dobra": fold, **params})
display(pd.DataFrame(choices))
plot_diagnostics(payload, X, y)

## 4. Salvar os resultados

Uma execução completa grava escores de todas as dobras, escolhas de hiperparâmetros e previsões externas em JSON. O notebook final verifica a compatibilidade dos dados, código e protocolo antes de comparar. Uma nova execução substitui o resultado desta família.

In [ ]:
if not SMOKE_TEST:
    print("Resultado salvo em:", save_results(payload))
else:
    print("Smoke test concluído; nenhum resultado final foi salvo.")

## 5. Conclusão

Preencha após executar:

- Qual pipeline teve o menor RMSE externo? Qual foi o ganho sobre a linear com afinidade?
- O ganho aparece de maneira consistente nas mesmas dobras?
- Há sinais de sobreajuste ou padrões restantes nos resíduos?
- Os hiperparâmetros são estáveis? O ganho compensa a complexidade e o custo?

**Resultado:** ainda não executado em modo completo. Evite tratar o menor erro observado entre muitas famílias como uma estimativa independente da seleção final.